# The N-Ball Transformer — Holcus Vision

**One Claim:** The unit-ball volume `V(n) = π^(n/2)/Γ(n/2+1)` is the
Cayley–Dickson layer transformer. Its first two step ratios are **exactly**
`π/2` — `V(2)/V(1) = V(4)/V(2) = π/2` — the sequence **breaks** at `ℍ→𝕆` where
`V(8)/V(4) = π²/12`, and the volume peaks at a **non-integer** dimension `n*`
satisfying `ψ(n*/2 + 1) = ln π`.

**Paper:** FourthAgePapers / NBallTransformer
**Engine:** `ValaQuenta/fixed_point.py`
**Data:** None external. Every value is computed from the definition.
**Wiki:** Written last.

---

## The computer science

The mathematics of `V(n)` is nineteenth-century and not in dispute. What this
paper is about is **computing it correctly**, and that is a CS problem with
three distinct parts:

1. **An exact identity.** `V(2)/V(1) = V(4)/V(2) = π/2` is *exact* — it holds in
   rational-plus-π symbolic arithmetic, with no rounding anywhere. This paper
   proves it symbolically rather than observing it numerically. A numerical
   coincidence to 15 digits and an algebraic identity are different claims, and
   only the second one is safe to build on.

2. **A floating-point implementation that is not exact.** The engine computes
   `V(n)` with `math.gamma`. The result differs from the exact value. This paper
   measures that gap **in units in the last place (ulp)** rather than calling it
   "equal to 15 decimal places", because ulp is the honest unit for the question
   *did the implementation return the correctly-rounded answer?*

3. **A root that is found by the wrong algorithm.** `V` peaks where
   `dV/dn = 0`, i.e. `ψ(n/2+1) = ln π`. The engine locates this by evaluating
   `digamma` on a 10,000-point grid and taking the `argmin` — a search whose
   accuracy is bounded by the grid spacing, not by the arithmetic. Replacing it
   with a bracketed root solve is both cheaper and about six orders of magnitude
   more accurate. That is an algorithm-selection result, and it is the most
   practically useful thing in this paper.

## Why "transformer"

Each Cayley–Dickson doubling ℝ→ℂ→ℍ→𝕆→𝕊 moves the dimension `1→2→4→8→16`.
`V(n)` evaluated along that sequence gives the volume carried at each layer,
and the *ratio* between consecutive layers is the gain of the transform. The
claim is that this gain is constant at `π/2` for the first two steps and then
stops being constant — the algebra loses a property (commutativity, then
associativity) at exactly the point the volume ratio stops repeating.

The paper reports the ratio. It does **not** claim to explain why the algebraic
loss and the ratio change coincide; that would be an interpretation and there is
no code here that tests it.

---

## Scope

**Claimed:** the two exact identities; the exact break value; the exact
recurrence; the ulp-level behaviour of the float implementation; the accuracy
and cost of grid search versus root finding.

**Not claimed:** any physical meaning of `n*`, any consequence for the algebra
tower beyond the arithmetic, and any statement about dimensions that are not
non-negative reals.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

from ValaQuenta import fixed_point as fp

import math
import numpy as np
import sympy as sy
import scipy.special as ss
import scipy.optimize as so

print('engine :', 'ValaQuenta/fixed_point.py')
print('python :', sys.version.split()[0])

In [ ]:
# Exact V(n) in rational/symbolic arithmetic -- no floating point at all.
def V_exact(k):
    """V(k) = pi^(k/2) / Gamma(k/2 + 1) as an exact sympy expression."""
    return sy.pi**sy.Rational(k, 2) / sy.gamma(sy.Rational(k, 2) + 1)

def V_float(x):
    """The engine's floating-point implementation."""
    return fp.v_nball(x)

In [ ]:
# The claim, stated once, in exact arithmetic.
for a, b in [(2, 1), (4, 2), (8, 4)]:
    r = sy.simplify(V_exact(a) / V_exact(b))
    same = sy.simplify(r - sy.pi/2) == 0
    print(f'V({a})/V({b}) = {r}      == pi/2 ? {same}')
print()
print('Run 01_predictions.ipynb next.')